# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

In [1]:
# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [2]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [3]:
#import os
#os.environ["PATH"] = f"/home/p2chung/.local/bin:{os.environ['PATH']}"

#!uv venv .venv --seed
#!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter
#!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
#print("Done.")

### Run the cell below every time to activate the installed environment. 

In [4]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [5]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = "data/public.jsonl"
MAX_TOKENS  = 32768

# Change this between runs:
OUTPUT_PATH = "results/run3_self_consistency_3x_200q.jsonl"
GENERATION_CONFIG = {
    "max_new_tokens": MAX_TOKENS,
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "repetition_penalty": 1.0,
    "do_sample": True,
}

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import sys
from pathlib import Path
from typing import Optional
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [6]:
# cutoff_rate = sum("\\boxed" not in r["response"] for r in saved_data) / len(saved_data)
# print(f"Cutoff rate (no boxed answer): {cutoff_rate:.2%}")

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [7]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition MIG 1g.24gb


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [9]:
PROMPT_VERSION = "concise"  # change to "concise" for runs 2, 3, 4

SYSTEM_PROMPT_MATH_STARTER = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)
SYSTEM_PROMPT_MCQ_STARTER = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)
SYSTEM_PROMPT_MATH_CONCISE = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, but be concise — no repetition, no over-explanation. "
    "Think about the problem type first, then apply the correct method. "
    "Your response MUST end with \\boxed{answer} as the absolute last thing you write. "
    "For multiple sub-answers in order, use \\boxed{a, b, c}. "
    "Never write anything after the boxed answer."
)
SYSTEM_PROMPT_MCQ_CONCISE = (
    "You are an expert mathematician. "
    "Read the problem and all options carefully. Eliminate wrong answers, then select the best one. "
    "Your response MUST end with \\boxed{X} where X is the letter only. "
    "Never write anything after the boxed answer."
)

SYSTEM_PROMPT_MATH = SYSTEM_PROMPT_MATH_STARTER if PROMPT_VERSION == "starter" else SYSTEM_PROMPT_MATH_CONCISE
SYSTEM_PROMPT_MCQ  = SYSTEM_PROMPT_MCQ_STARTER  if PROMPT_VERSION == "starter" else SYSTEM_PROMPT_MCQ_CONCISE

from typing import Optional

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [10]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [11]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=16384,
    trust_remote_code=True,
    disable_log_stats=True,
)

tokenizer = llm.get_tokenizer()
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded.")

INFO 05-20 19:40:27 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-20 19:40:42 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.


WARNING 05-20 19:40:42 [nixl_utils.py:34] NIXL is not available


WARNING 05-20 19:40:42 [nixl_utils.py:44] NIXL agent config is not available


...

Capturing CUDA graphs (decode, FULL):  60%|██████    | 21/35 [00:01<00:00, 21.03it/s]

Capturing CUDA graphs (decode, FULL):  77%|███████▋  | 27/35 [00:01<00:00, 22.76it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 21.07it/s]


(EngineCore pid=472) INFO 05-20 19:41:57 [gpu_model_runner.py:6133] Graph capturing finished in 6 secs, took 0.40 GiB
(EngineCore pid=472) INFO 05-20 19:41:57 [gpu_worker.py:599] CUDA graph pool memory: 0.4 GiB (actual), 0.35 GiB (estimated), difference: 0.05 GiB (12.7%).
(EngineCore pid=472) INFO 05-20 19:41:57 [core.py:299] init engine (profile, create kv cache, warmup model) took 46.40 s (compilation: 28.04 s)


(EngineCore pid=472) INFO 05-20 19:41:59 [vllm.py:840] Asynchronous scheduling is enabled.
(EngineCore pid=472) INFO 05-20 19:41:59 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model loaded.


In [12]:
# !.venv/bin/python -m pip install accelerate


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [13]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [14]:
from pathlib import Path
import json
from collections import Counter

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 3  # run each question 3 times, take majority

items = data[:200]

# Build prompts (repeat each N_SAMPLES times)
prompts = []
for item in items:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for _ in range(N_SAMPLES):
        prompts.append((item, prompt_text))

sampling_params = SamplingParams(
    max_tokens=GENERATION_CONFIG["max_new_tokens"],
    temperature=GENERATION_CONFIG.get("temperature", 0.6),
    top_p=GENERATION_CONFIG.get("top_p", 0.95),
    top_k=GENERATION_CONFIG.get("top_k", 20),
    repetition_penalty=GENERATION_CONFIG.get("repetition_penalty", 1.0),
)

print(f"Generating {N_SAMPLES} responses per question for {len(items)} questions ({len(prompts)} total)...")
outputs = llm.generate([p for _, p in prompts], sampling_params)

# Group responses by question and take majority
responses_by_id = {}
for (item, _), output in zip(prompts, outputs):
    qid = item.get("id")
    response = output.outputs[0].text.strip()
    if qid not in responses_by_id:
        responses_by_id[qid] = {"item": item, "responses": []}
    responses_by_id[qid]["responses"].append(response)

def extract_final_answer(response, is_mcq):
    if is_mcq:
        m = re.search(r"\\boxed\{([A-Za-z])\}", response)
        if m:
            return m.group(1).upper()
        matches = re.findall(r"\b([A-Z])\b", response.upper())
        return matches[-1] if matches else ""
    else:
        m = re.search(r"\\boxed\{([^}]+)\}", response)
        return m.group(1).strip() if m else ""

# Majority vote
responses = []
with open(out_path, "w") as f:
    for qid, data_item in responses_by_id.items():
        item = data_item["item"]
        all_responses = data_item["responses"]
        is_mcq = bool(item.get("options"))
        
        # Extract answers from all samples
        answers = [extract_final_answer(r, is_mcq) for r in all_responses]
        
        # Majority vote
        if answers:
            majority_answer = Counter(answers).most_common(1)[0][0]
        else:
            majority_answer = ""
        
        # Use the response that matches majority answer, or first response
        best_response = next(
            (r for r in all_responses if extract_final_answer(r, is_mcq) == majority_answer),
            all_responses[0]
        )
        responses.append(best_response)
        
        record = {
            "id": item.get("id"),
            "question": item["question"],
            "response": best_response,
            "answer": item.get("answer"),
            "options": item.get("options"),
            "all_answers": answers,
            "majority_answer": majority_answer,
        }
        f.write(json.dumps(record) + "\n")

boxed_rate = sum("\\boxed" in r for r in responses) / len(responses)
avg_len = sum(len(r) for r in responses) / len(responses)
print(f"\nBoxed answer rate: {boxed_rate:.2%}")
print(f"Average response length: {avg_len:.0f} chars")
print(f"Saved {len(responses)} majority-voted responses to {out_path}")

Generating 3 responses per question for 200 questions (600 total)...


Rendering prompts:   0%|          | 0/600 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/600 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Boxed answer rate: 93.00%
Average response length: 11396 chars
Saved 200 majority-voted responses to results/run3_self_consistency_3x_200q.jsonl


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [15]:
import json

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# Load responses from saved file instead of memory
saved_data = []
with open(OUTPUT_PATH) as f:
    for line in f:
        saved_data.append(json.loads(line))

# Rebuild results from saved file
results = []
for record in saved_data:
    item = record
    is_mcq = bool(item.get("options"))
    gold = item["answer"]
    response = item["response"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "correct": correct,
    })

print(f"Loaded and scored {len(results)} results from {OUTPUT_PATH}")

Loaded and scored 200 results from results/run3_self_consistency_3x_200q.jsonl


## 8. Summary

Print accuracy broken down by question type.

In [16]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   51 /   68  (75.00%)
  Free-form  :   73 /  132  (55.30%)
  Overall    :  124 /  200  (62.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [17]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 200 records to results/run3_self_consistency_3x_200q.jsonl


In [18]:
boxed_rate = sum("\\boxed" in r["response"] for r in saved_data) / len(saved_data)
print(f"Boxed rate: {boxed_rate:.2%}")

Boxed rate: 93.00%


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!